# Lesson 9: Model Interpretation

Understanding what your model learned.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Permutation Importance

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Built-in vs. Permutation
result = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)

comparison = pd.DataFrame({
    'feature': data.feature_names,
    'impurity': rf.feature_importances_,
    'permutation': result.importances_mean,
})

top_features = comparison.sort_values('permutation', ascending=False).head(10)
print(top_features.to_string(index=False))

In [ ]:
top_features.plot(x='feature', y=['impurity', 'permutation'], kind='barh', figsize=(10, 5))
plt.title('Impurity vs. Permutation Importance')
plt.tight_layout()
plt.show()

## 2. Partial Dependence Plots

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
PartialDependenceDisplay.from_estimator(
    rf, X_test, ['worst radius', 'worst concave points'],
    grid_resolution=20, ax=ax
)
plt.suptitle('Partial Dependence: Breast Cancer')
plt.tight_layout()
plt.show()

## 3. PDP for California Housing

In [ ]:
housing = fetch_california_housing()
X_h, y_h = housing.data, housing.target
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(X_h, y_h, test_size=0.2, random_state=42)

rf_h = RandomForestRegressor(n_estimators=100, random_state=42)
rf_h.fit(Xh_tr, yh_tr)

fig, ax = plt.subplots(figsize=(10, 4))
PartialDependenceDisplay.from_estimator(
    rf_h, Xh_te, ['MedInc', 'AveOccup'],
    grid_resolution=20, ax=ax
)
plt.suptitle('Partial Dependence: California Housing')
plt.tight_layout()
plt.show()

## 4. Permutation Importance with Repeats

In [ ]:
result_h = permutation_importance(rf_h, Xh_te, yh_te, n_repeats=10, random_state=42)

sorted_idx = result_h.importances_mean.argsort()[::-1]

plt.figure(figsize=(8, 5))
plt.barh([housing.feature_names[i] for i in sorted_idx],
         result_h.importances_mean[sorted_idx],
         xerr=result_h.importances_std[sorted_idx])
plt.xlabel('Permutation Importance')
plt.title('California Housing — Permutation Importance')
plt.tight_layout()
plt.show()

## 5. Biotechnology: Gene Discovery

In [ ]:
n, n_g = 300, 100
X_genes = np.random.randn(n, n_g)
y_disease = (X_genes[:, 0]*0.5 + X_genes[:, 15]*0.3 - X_genes[:, 42]*0.4 > 0).astype(int)

rf_gene = RandomForestClassifier(n_estimators=100, random_state=42)
rf_gene.fit(X_genes, y_disease)

perm = permutation_importance(rf_gene, X_genes, y_disease, n_repeats=5, random_state=42)
top = np.argsort(perm.importances_mean)[::-1][:5]
print("Top genes:")
for g in top:
    print(f"  Gene {g}: importance = {perm.importances_mean[g]:.4f}")

## 6. Exercises

### Level 1
Why is permutation importance more reliable than impurity-based importance?

### Level 2
Train RF on California Housing, compute permutation importance, plot top 5.

### Level 3
Features A and B are correlated (r=0.95). Permutation importance for both is low. Remove A and B becomes important. What's happening?

In [ ]:
# Level 2 code

## 7. Coding Challenge

Write `compare_importances(model, X_val, y_val, feature_names)` returning a comparison DataFrame.

In [ ]:
def compare_importances(model, X_val, y_val, feature_names):
    impurity = model.feature_importances_
    perm = permutation_importance(model, X_val, y_val, n_repeats=10, random_state=42)
    df = pd.DataFrame({
        'feature': feature_names,
        'impurity': impurity,
        'permutation': perm.importances_mean,
        'perm_std': perm.importances_std
    }).sort_values('permutation', ascending=False)
    return df

comparison_df = compare_importances(rf, X_test, y_test, data.feature_names)
print(comparison_df.head(10).to_string(index=False))

## Summary

- Permutation importance: shuffle → measure score drop
- Partial dependence: how feature affects predictions
- SHAP/LIME for local explanations
- Always compare multiple importance methods
- Domain expertise is essential
- Interpretability builds trust and enables discovery